# Dense Piano Transformer + PianoREMIBPE (Kaggle Dual T4)

This notebook trains `DensePianoTransformer` on solo-piano MIDI tokenized with `PianoREMIBPE`.

The architecture is a decoder-only transformer with dense softmax attention in every layer. It uses local causal windows that widen with depth: early layers focus on nearby note grammar, middle layers see longer phrases, and upper layers attend over a 2048-token window plus learned global anchor tokens for stable long-range structure. This keeps the model simpler than hybrid SSM or recurrent mixtures while still giving upper layers enough context for musical form.

`PianoREMIBPE` keeps velocity in the autoregressive stream as `NOTE_ON -> VELOCITY -> DURATION`, so dynamics are learned as part of the musical language instead of being stripped into side data. Sustain pedal events are first-class tokens. The BPE vocabulary is trained over REMI-style integer tokens and defaults to about 30K tokens.

Kaggle dual T4 setup notes: training is launched with PyTorch DDP (`torchrun`, NCCL backend). The script requests bfloat16 first, but includes a runtime guard that falls back to float16 when the GPU does not support bf16. Context length is pinned to 8192, batch size starts at 1 per GPU, and gradient checkpointing is enabled for every transformer layer.


In [ ]:
from pathlib import Path
import getpass
import importlib
import json
import os
import subprocess
import sys

import torch

RUN_TRAINING = True


def _ensure_packages() -> None:
    required = ["pretty_midi", "safetensors", "symusic", "numpy", "tqdm", "huggingface_hub", "miditok"]
    missing = [pkg for pkg in required if importlib.util.find_spec(pkg) is None]
    if missing:
        subprocess.check_call([
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "--disable-pip-version-check",
            *missing,
        ])


def _find_root() -> Path:
    probes = [
        Path.cwd().resolve(),
        Path("/kaggle/working/piano_midi_model"),
        Path("/kaggle/input/magic_midi"),
        Path("/kaggle/input/magic_midi/piano_midi_model"),
        Path("/kaggle/input/magic-midi"),
        Path("/kaggle/input"),
    ]
    anchor = Path("training/train_dense_piano_transformer_ddp.py")
    for probe in probes:
        if not probe.exists():
            continue
        for candidate in [probe, *probe.parents]:
            if (candidate / anchor).exists():
                return candidate
        for candidate in probe.rglob("train_dense_piano_transformer_ddp.py"):
            return candidate.resolve().parents[1]
    raise FileNotFoundError("Project root not found. Add the repository as a Kaggle input dataset.")


_ensure_packages()
PROJECT_ROOT = _find_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from hf_sync import normalize_hf_repo_id, resolve_latest_hf_checkpoint

OUTPUT_DIR = Path("/kaggle/working/dense_piano_remi_bpe")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

HF_TOKEN = (
    os.environ.get("HF_TOKEN", "").strip()
    or os.environ.get("HUGGINGFACE_HUB_TOKEN", "").strip()
    or getpass.getpass("Paste Hugging Face token for checkpoint sync: ").strip()
)
if not HF_TOKEN:
    raise RuntimeError("HF_TOKEN is required for Hugging Face checkpoint sync.")
HF_SYNC_REPO_ID = normalize_hf_repo_id(
    os.environ.get("HF_SYNC_REPO_ID", "").strip() or "Chickaboo/Pulse88-0.1-83M-Alpha"
)
HF_SYNC_PRIVATE = str(os.environ.get("HF_SYNC_PRIVATE", "1")).strip().lower() in {"1", "true", "yes", "on"}
HF_SYNC_EVERY_N_STEPS = int(os.environ.get("HF_SYNC_EVERY_N_STEPS", "500"))
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGINGFACE_HUB_TOKEN"] = HF_TOKEN
os.environ["HF_SYNC_REPO_ID"] = HF_SYNC_REPO_ID
os.environ["HF_SYNC_PRIVATE"] = "1" if HF_SYNC_PRIVATE else "0"
os.environ["HF_SYNC_EVERY_N_STEPS"] = str(HF_SYNC_EVERY_N_STEPS)
print(f"PROJECT_ROOT={PROJECT_ROOT}")
print(f"OUTPUT_DIR={OUTPUT_DIR}")
print(f"HF_SYNC_REPO_ID={HF_SYNC_REPO_ID}")
print(f"HF_SYNC_EVERY_N_STEPS={HF_SYNC_EVERY_N_STEPS}")
print(f"CUDA devices={torch.cuda.device_count() if torch.cuda.is_available() else 0}")


In [ ]:
# Locate PianoREMIBPE tokenized data. Override these paths with environment variables if needed.
PRETOKENIZED_ROOT = Path(os.environ.get("PRETOKENIZED_ROOT", "")).expanduser() if os.environ.get("PRETOKENIZED_ROOT") else None
PRETOKENIZED_MANIFEST = Path(os.environ.get("PRETOKENIZED_MANIFEST", "")).expanduser() if os.environ.get("PRETOKENIZED_MANIFEST") else None
TOKENIZER_PATH = Path(os.environ.get("PIANO_REMI_BPE_TOKENIZER", "")).expanduser() if os.environ.get("PIANO_REMI_BPE_TOKENIZER") else None


def _first_existing(candidates):
    for path in candidates:
        if path is not None and path.exists():
            return path.resolve()
    return None


def _manifest_root(manifest_path: Path) -> Path:
    manifest_path = manifest_path.resolve()
    if manifest_path.parent.name == "metadata":
        return manifest_path.parent.parent.resolve()
    return manifest_path.parent.resolve()


def _has_manifest(folder: Path) -> bool:
    return (folder / "metadata" / "manifest.json").exists() or (folder / "manifest.json").exists()


if PRETOKENIZED_ROOT is None:
    root_candidates = []
    kaggle_input = Path("/kaggle/input")
    explicit_roots = [
        Path("/kaggle/input/datasets/chickaboomcmurtrie/godzilla-piano-remi-bpe-500k/godzilla_piano_remi_bpe_500k"),
    ]
    preferred_names = [
        "godzilla_piano_remi_bpe_500k",
        "godzilla-piano-remi-bpe-500k",
    ]
    for folder in explicit_roots:
        if folder.is_dir() and _has_manifest(folder):
            root_candidates.append(folder)
    if kaggle_input.exists():
        for name in preferred_names:
            folder = kaggle_input / name
            if folder.is_dir() and _has_manifest(folder):
                root_candidates.append(folder)
            if folder.is_dir():
                for manifest in list(folder.glob("*/metadata/manifest.json")) + list(folder.glob("*/manifest.json")):
                    root_candidates.append(_manifest_root(manifest))
        for folder in sorted(kaggle_input.iterdir(), key=lambda p: p.name.lower()):
            if folder in root_candidates:
                continue
            if folder.is_dir() and _has_manifest(folder):
                root_candidates.append(folder)
        if not root_candidates:
            manifest_patterns = [
                "*/*/metadata/manifest.json",
                "*/*/manifest.json",
                "datasets/*/*/*/metadata/manifest.json",
                "datasets/*/*/*/manifest.json",
            ]
            for pattern in manifest_patterns:
                for manifest in kaggle_input.glob(pattern):
                    root_candidates.append(_manifest_root(manifest))
    PRETOKENIZED_ROOT = root_candidates[0].resolve() if root_candidates else None

if PRETOKENIZED_MANIFEST is None and PRETOKENIZED_ROOT is not None:
    PRETOKENIZED_MANIFEST = _first_existing([
        PRETOKENIZED_ROOT / "metadata" / "manifest.json",
        PRETOKENIZED_ROOT / "manifest.json",
    ])

if TOKENIZER_PATH is None and PRETOKENIZED_ROOT is not None:
    TOKENIZER_PATH = _first_existing([
        PRETOKENIZED_ROOT / "metadata" / "piano_remi_bpe_tokenizer.json",
        PRETOKENIZED_ROOT / "piano_remi_bpe_tokenizer.json",
    ])

if PRETOKENIZED_ROOT is None or PRETOKENIZED_MANIFEST is None:
    kaggle_listing = []
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        kaggle_listing = [str(p) for p in sorted(kaggle_input.iterdir(), key=lambda x: x.name.lower())[:50]]
    raise FileNotFoundError(
        "Could not locate PianoREMIBPE tokenized data. "
        "Set PRETOKENIZED_ROOT and PRETOKENIZED_MANIFEST. "
        f"Visible /kaggle/input entries: {kaggle_listing}"
    )

print(f"PRETOKENIZED_ROOT={PRETOKENIZED_ROOT}")
print(f"PRETOKENIZED_DATASET_NAME={PRETOKENIZED_ROOT.name}")
print(f"PRETOKENIZED_MANIFEST={PRETOKENIZED_MANIFEST}")
print(f"TOKENIZER_PATH={TOKENIZER_PATH if TOKENIZER_PATH else 'not found; trainer will create a default 30K tokenizer shell'}")


In [ ]:
# Launch DDP training. Defaults are sized for Kaggle dual T4: batch 1/GPU, grad accumulation 16, 8192-token context.
DDP_SCRIPT = PROJECT_ROOT / "training" / "train_dense_piano_transformer_ddp.py"
if not DDP_SCRIPT.exists():
    raise FileNotFoundError(DDP_SCRIPT)

NUM_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0
NPROC = max(1, min(2, int(NUM_GPUS)))
remote_resume = resolve_latest_hf_checkpoint(
    repo_id=HF_SYNC_REPO_ID,
    cache_root=OUTPUT_DIR,
    token=HF_TOKEN,
    repo_type="model",
)
if remote_resume is not None:
    print(f"Resolved Hugging Face resume checkpoint: {remote_resume}")
else:
    print("No Hugging Face resume checkpoint found; trainer will fall back to local auto-resume if available.")

cmd = [
    sys.executable, "-m", "torch.distributed.run", "--standalone", "--nproc_per_node", str(NPROC), str(DDP_SCRIPT),
    "--pretokenized_manifest", str(PRETOKENIZED_MANIFEST),
    "--pretokenized_root", str(PRETOKENIZED_ROOT),
    "--output_dir", str(OUTPUT_DIR),
    "--vocab_size", "30000",
    "--seed_length", "4096",
    "--continuation_length", "4096",
    "--max_sequence_length", "8192",
    "--d_model", "576",
    "--n_layers", "12",
    "--num_attention_heads", "9",
    "--head_dim", "64",
    "--global_anchor_count", "256",
    "--batch_size", "1",
    "--grad_accumulation_steps", "16",
    "--learning_rate", "2e-4",
    "--label_smoothing", "0.02",
    "--amp_dtype", "bfloat16",
    "--save_every_n_steps", "500",
    "--hf_sync_repo_id", HF_SYNC_REPO_ID,
    "--hf_sync_every_n_steps", str(HF_SYNC_EVERY_N_STEPS),
]
if HF_SYNC_PRIVATE:
    cmd.append("--hf_sync_private")
if remote_resume is not None:
    cmd.extend(["--resume_from_checkpoint", str(remote_resume)])
if TOKENIZER_PATH is not None:
    cmd.extend(["--tokenizer_path", str(TOKENIZER_PATH)])

print(" ".join(cmd))
if RUN_TRAINING:
    subprocess.check_call(cmd, cwd=str(PROJECT_ROOT))
else:
    print("RUN_TRAINING=False; command not executed.")


In [ ]:
# Optional quick result inspection
result_path = OUTPUT_DIR / "dense_piano_transformer_ddp_result.json"
if result_path.exists():
    payload = json.loads(result_path.read_text(encoding="utf-8"))
    history = payload.get("result", {}).get("history", {})
    step_checkpoints = history.get("step_checkpoints", []) if isinstance(history, dict) else []
    print(json.dumps({
        "params": payload.get("result", {}).get("params"),
        "checkpoint_dir": payload.get("result", {}).get("checkpoint_dir"),
        "amp_dtype": payload.get("distributed", {}).get("amp_dtype"),
        "train_pieces": payload.get("data", {}).get("train_pieces"),
        "val_pieces": payload.get("data", {}).get("val_pieces"),
        "hf_sync": payload.get("hf_sync", {}),
        "latest_step_checkpoint": step_checkpoints[-1] if step_checkpoints else None,
    }, indent=2))
else:
    print(f"No result file yet: {result_path}")


In [ ]:
# Optional lightweight artifact browser for Kaggle output files.
for path in sorted(OUTPUT_DIR.rglob("*"), key=lambda p: p.stat().st_mtime if p.is_file() else 0, reverse=True)[:50]:
    if path.is_file() and path.suffix.lower() in {".pt", ".safetensors", ".json", ".mid", ".txt"}:
        print(path)
